# Test the fine-tuned Gemma3 1B model

Loads the model produced by `finetune.py` (saved to `/workspace/finetuned.keras`) and checks the instruction-following behavior with `generate()`, using the same prompt format the model was fine-tuned on:

```
[instruction]
{instruction}[end]
[response]
```

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

import keras
import keras_hub

## Load the fine-tuned model

Primary path: load the full native `.keras` model. `keras_hub` must be imported first so its custom layers are registered for deserialization.

In [ ]:
gemma_lm = keras.models.load_model("/workspace/finetuned.keras")

Fallback (if the `.keras` file is unavailable): rebuild the preset, re-enable LoRA, and load the backup weights.

```python
gemma_lm = keras_hub.models.CausalLM.from_preset("gemma3_1b", dtype="float32")
gemma_lm.backbone.enable_lora(rank=8)
gemma_lm.load_weights("/workspace/finetuned.weights.h5")
```

## Generate responses

Same instruction prompts used to validate the fine-tune in the chapter notebook.

In [ ]:
gemma_lm.generate(
    "[instruction]\nHow can I make brownies?[end]\n"
    "[response]\n",
    max_length=512,
)

In [ ]:
gemma_lm.generate(
    "[instruction]\nWhat is a proper noun?[end]\n"
    "[response]\n",
    max_length=512,
)

In [ ]:
gemma_lm.generate(
    "[instruction]\nWho is the 542nd president of the United States?[end]\n"
    "[response]\n",
    max_length=512,
)